# LeetCode 234 — Palindrome Linked List bằng Python

- Link đề: https://leetcode.com/problems/palindrome-linked-list/
- Độ khó: **Easy** · Chủ đề: Linked List, Two Pointers (nhanh/chậm), Stack, Recursion

Chữ ký hàm LeetCode:

```python
# Definition for singly-linked list.
# class ListNode:
#     def __init__(self, val=0, next=None):
#         self.val = val
#         self.next = next
class Solution:
    def isPalindrome(self, head: ListNode | None) -> bool:

```

**Mục tiêu học**

1. Hiểu danh sách liên kết đơn (singly linked list): node, con trỏ `next`, duyệt bằng `while`.
2. Giải bài bằng cách dễ nhất: chép giá trị ra list Python rồi so sánh với bản đảo.
3. Học hai kỹ thuật cốt lõi của linked list: **con trỏ nhanh/chậm** để tìm giữa và **đảo ngược danh sách tại chỗ**.
4. Ghép chúng để đạt Follow-up: `O(n)` thời gian, `O(1)` bộ nhớ phụ.
5. Biết thêm cách stack và cách đệ quy, so sánh ưu nhược.

## 1. Đề bài và ví dụ

Cho `head` của một danh sách liên kết đơn. Trả `True` nếu dãy giá trị là **palindrome** (đọc xuôi và ngược như nhau).

| Danh sách | Kết quả |
|---|---|
| `1 -> 2 -> 2 -> 1` | `True` |
| `1 -> 2` | `False` |
| `1 -> 2 -> 3 -> 2 -> 1` | `True` (độ dài lẻ, giữa là 3) |
| `7` | `True` |

**Constraints:** số node trong `[1, 10^5]`; `0 <= Node.val <= 9`.

**Follow up:** giải trong `O(n)` thời gian và `O(1)` bộ nhớ?

Khó khăn chính: với mảng ta đọc ngược dễ dàng bằng index, còn linked list **chỉ đi được một chiều** (tiến theo `next`).

## 2. Nền tảng: `ListNode` và các hàm tiện ích

LeetCode tự định nghĩa `ListNode`; trong notebook ta phải tự viết để chạy độc lập. Mỗi node có:

- `val`: giá trị;
- `next`: tham chiếu tới node kế tiếp (hoặc `None` nếu là node cuối).

In [1]:
class ListNode:
    def __init__(self, val=0, next=None):
        self.val = val
        self.next = next

    def __repr__(self):
        return f"ListNode({self.val})"

a = ListNode(1)
b = ListNode(2)
a.next = b            # nối a -> b
print(a, "->", a.next, "->", a.next.next)

ListNode(1) -> ListNode(2) -> None


Viết tay từng node rất mệt, nên ta tạo hàm `build(values)` dựng danh sách từ list Python và `to_list(head)` đọc ngược lại để in.

In [2]:
def build(values: list[int]) -> ListNode | None:
    dummy = ListNode()          # node giả đứng trước head, giúp code gọn
    tail = dummy
    for v in values:
        tail.next = ListNode(v)
        tail = tail.next
    return dummy.next

def to_list(head: ListNode | None) -> list[int]:
    out = []
    node = head
    while node is not None:     # duyệt cho tới hết danh sách
        out.append(node.val)
        node = node.next
    return out

def show(head):
    return " -> ".join(map(str, to_list(head))) or "(rỗng)"

h = build([1, 2, 2, 1])
print(to_list(h))
print(show(h))

[1, 2, 2, 1]
1 -> 2 -> 2 -> 1


Giải thích:

- **Dummy node**: node giả để không phải xử lý riêng "danh sách đang rỗng" khi nối node đầu tiên.
- `while node is not None:` là mẫu duyệt linked list chuẩn: xử lý `node`, rồi `node = node.next`.
- `map(str, ...)` đổi từng số thành chuỗi; `" -> ".join(...)` nối lại.
- `ListNode | None` là type hint "ListNode hoặc None" (Python 3.10+).

## 3. Cách 1 — Chép ra list rồi so sánh (baseline)

Bài palindrome trên mảng thì quá dễ: `vals == vals[::-1]`. Vậy cứ chép linked list sang mảng!

In [3]:
def is_palindrome_copy(head: ListNode | None) -> bool:
    vals = []
    node = head
    while node:
        vals.append(node.val)
        node = node.next
    return vals == vals[::-1]

for arr in [[1, 2, 2, 1], [1, 2], [1, 2, 3, 2, 1], [7]]:
    print(arr, "->", is_palindrome_copy(build(arr)))

[1, 2, 2, 1] -> True
[1, 2] -> False
[1, 2, 3, 2, 1] -> True
[7] -> True


- `vals[::-1]` là slicing với bước `-1`: tạo bản sao đảo ngược.
- Time `O(n)`, Space `O(n)` cho list `vals`.

Có thể so sánh bằng **hai con trỏ** trên mảng thay vì tạo bản đảo:

In [4]:
def is_palindrome_copy_two_pointers(head):
    vals = to_list(head)
    i, j = 0, len(vals) - 1
    while i < j:
        if vals[i] != vals[j]:
            return False
        i += 1
        j -= 1
    return True

print(is_palindrome_copy_two_pointers(build([1, 2, 3, 2, 1])),
      is_palindrome_copy_two_pointers(build([1, 2, 3, 1])))

True False


### Nút thắt

Cả hai đều cần `O(n)` bộ nhớ phụ. Follow-up yêu cầu `O(1)`. Ta không thể đi ngược trên linked list... **trừ khi ta tự đảo chiều nửa sau**.

Kế hoạch `O(1)`:

1. Tìm **giữa** danh sách (con trỏ nhanh/chậm).
2. **Đảo ngược** nửa sau tại chỗ.
3. So sánh nửa đầu với nửa sau đã đảo, từng node một.
4. (Lịch sự) đảo lại nửa sau để trả danh sách về nguyên trạng.

Ta học riêng từng kỹ thuật trước.

## 4. Kỹ thuật 1 — Con trỏ nhanh/chậm tìm giữa

- `slow` đi 1 bước mỗi vòng, `fast` đi 2 bước.
- Khi `fast` tới cuối, `slow` ở giữa (vì nó đi được một nửa quãng đường).

In [5]:
def find_middle(head):
    slow = fast = head
    while fast and fast.next:
        slow = slow.next
        fast = fast.next.next
    return slow

for arr in [[1, 2, 3, 4, 5], [1, 2, 3, 4], [1, 2], [1]]:
    print(f"{arr} -> giữa = {find_middle(build(arr)).val}")

[1, 2, 3, 4, 5] -> giữa = 3
[1, 2, 3, 4] -> giữa = 3
[1, 2] -> giữa = 2
[1] -> giữa = 1


Quan sát:

- Độ dài **lẻ** `[1,2,3,4,5]` → `slow` ở `3` (đúng giữa).
- Độ dài **chẵn** `[1,2,3,4]` → `slow` ở `3` = **đầu nửa sau**.

Điều kiện `while fast and fast.next` bảo vệ ta khỏi lỗi `AttributeError: 'NoneType' object has no attribute 'next'`: chỉ nhảy 2 bước khi cả `fast` và `fast.next` tồn tại.

In [6]:
def trace_middle(values):
    head = build(values)
    slow = fast = head
    step = 0
    print(f"Danh sách {values}")
    while fast and fast.next:
        slow = slow.next
        fast = fast.next.next
        step += 1
        print(f"  vòng {step}: slow={slow.val}, fast={fast.val if fast else None}")
    print(f"  dừng -> slow ở {slow.val}")

trace_middle([1, 2, 3, 4, 5])
trace_middle([1, 2, 2, 1])

Danh sách [1, 2, 3, 4, 5]
  vòng 1: slow=2, fast=3
  vòng 2: slow=3, fast=5
  dừng -> slow ở 3
Danh sách [1, 2, 2, 1]
  vòng 1: slow=2, fast=2
  vòng 2: slow=2, fast=None
  dừng -> slow ở 2


## 5. Kỹ thuật 2 — Đảo ngược linked list tại chỗ

Dùng 3 biến: `prev` (phần đã đảo), `cur` (node đang xét), `nxt` (lưu tạm node kế để không bị "lạc").

Mỗi vòng:

```python
nxt = cur.next      # 1. nhớ đường đi tiếp
cur.next = prev     # 2. quay đầu mũi tên
prev = cur          # 3. prev tiến lên
cur = nxt           # 4. cur tiến lên
```

In [7]:
def reverse_list(head):
    prev = None
    cur = head
    while cur:
        nxt = cur.next
        cur.next = prev
        prev = cur
        cur = nxt
    return prev            # prev là head mới

print(show(reverse_list(build([1, 2, 3, 4]))))

4 -> 3 -> 2 -> 1


In [8]:
def trace_reverse(values):
    cur = build(values)
    prev = None
    step = 0
    while cur:
        nxt = cur.next
        cur.next = prev
        prev = cur
        cur = nxt
        step += 1
        print(f"vòng {step}: phần đã đảo = {show(prev):<18} | phần còn lại = {show(cur)}")
    return prev

trace_reverse([1, 2, 3, 4])

vòng 1: phần đã đảo = 1                  | phần còn lại = 2 -> 3 -> 4
vòng 2: phần đã đảo = 2 -> 1             | phần còn lại = 3 -> 4
vòng 3: phần đã đảo = 3 -> 2 -> 1        | phần còn lại = 4
vòng 4: phần đã đảo = 4 -> 3 -> 2 -> 1   | phần còn lại = (rỗng)


ListNode(4)

Nếu quên bước 1 (`nxt = cur.next`), khi gán `cur.next = prev` ta mất đường tới phần còn lại của danh sách.

## 6. Lời giải chính — Tìm giữa + đảo nửa sau + so sánh

Ghép lại:

1. `mid = find_middle(head)`.
2. `second = reverse_list(mid)` — đầu của nửa sau đã đảo.
3. Duyệt song song `p1 = head`, `p2 = second` **chừng nào `p2` còn**, so `p1.val` với `p2.val`.
4. Khôi phục: `reverse_list(second)`.

Vì sao chỉ dừng theo `p2`? Nửa sau (tính từ `slow`) có độ dài `≤` nửa đầu + 1, và với độ dài lẻ node giữa nằm trong nửa sau, được so với chính nó (hoặc không ảnh hưởng). Xem trace dưới để thấy rõ.

In [9]:
def is_palindrome_reverse_half(head: ListNode | None) -> bool:
    if head is None or head.next is None:
        return True
    # 1. tìm giữa
    slow = fast = head
    while fast and fast.next:
        slow = slow.next
        fast = fast.next.next
    # 2. đảo nửa sau
    second = reverse_list(slow)
    # 3. so sánh
    p1, p2 = head, second
    result = True
    while p2:
        if p1.val != p2.val:
            result = False
            break
        p1 = p1.next
        p2 = p2.next
    # 4. khôi phục danh sách
    reverse_list(second)
    return result

for arr in [[1, 2, 2, 1], [1, 2], [1, 2, 3, 2, 1], [7], [1, 2, 3, 1]]:
    h = build(arr)
    print(arr, "->", is_palindrome_reverse_half(h), "| danh sách sau khi gọi:", to_list(h))

[1, 2, 2, 1] -> True | danh sách sau khi gọi: [1, 2, 2, 1]
[1, 2] -> False | danh sách sau khi gọi: [1, 2]
[1, 2, 3, 2, 1] -> True | danh sách sau khi gọi: [1, 2, 3, 2, 1]
[7] -> True | danh sách sau khi gọi: [7]
[1, 2, 3, 1] -> False | danh sách sau khi gọi: [1, 2, 3, 1]


Cột cuối cho thấy danh sách được trả về **nguyên vẹn** — quan trọng nếu người gọi còn dùng tiếp.

## 7. Trace chi tiết

Ví dụ `1 -> 2 -> 3 -> 2 -> 1` (độ dài lẻ):

| Giai đoạn | Trạng thái |
|---|---|
| Tìm giữa | `slow` dừng ở `3` |
| Đảo từ `3` | nửa đầu `1 -> 2 -> 3`, nửa sau đảo `1 -> 2 -> 3` (node 3 dùng chung, `2.next` vẫn trỏ 3) |
| So sánh | (1,1) ✓, (2,2) ✓, (3,3) ✓ → `p2 = None` dừng |
| Kết quả | `True` |

Ví dụ `1 -> 2 -> 2 -> 1` (chẵn): `slow` ở node `2` thứ hai; nửa sau đảo `1 -> 2`; so (1,1), (2,2) → `True`.

In [10]:
def trace_palindrome(values):
    head = build(values)
    print(f"Danh sách: {show(head)}")
    slow = fast = head
    while fast and fast.next:
        slow = slow.next
        fast = fast.next.next
    print(f"  giữa (slow) = {slow.val}")
    second = reverse_list(slow)
    print(f"  nửa sau sau khi đảo: {show(second)}")
    p1, p2, ok = head, second, True
    while p2:
        same = p1.val == p2.val
        print(f"  so p1={p1.val} với p2={p2.val} -> {'khớp' if same else 'KHÁC -> False'}")
        if not same:
            ok = False
            break
        p1, p2 = p1.next, p2.next
    reverse_list(second)
    print(f"  kết quả: {ok}; khôi phục: {show(head)}\n")
    return ok

trace_palindrome([1, 2, 3, 2, 1])
trace_palindrome([1, 2, 2, 1])
trace_palindrome([1, 2, 3, 3, 1])

Danh sách: 1 -> 2 -> 3 -> 2 -> 1
  giữa (slow) = 3
  nửa sau sau khi đảo: 1 -> 2 -> 3
  so p1=1 với p2=1 -> khớp
  so p1=2 với p2=2 -> khớp
  so p1=3 với p2=3 -> khớp
  kết quả: True; khôi phục: 1 -> 2 -> 3 -> 2 -> 1

Danh sách: 1 -> 2 -> 2 -> 1
  giữa (slow) = 2
  nửa sau sau khi đảo: 1 -> 2
  so p1=1 với p2=1 -> khớp
  so p1=2 với p2=2 -> khớp
  kết quả: True; khôi phục: 1 -> 2 -> 2 -> 1

Danh sách: 1 -> 2 -> 3 -> 3 -> 1
  giữa (slow) = 3
  nửa sau sau khi đảo: 1 -> 3 -> 3
  so p1=1 với p2=1 -> khớp
  so p1=2 với p2=3 -> KHÁC -> False
  kết quả: False; khôi phục: 1 -> 2 -> 3 -> 3 -> 1



False

`p1, p2 = p1.next, p2.next` là **gán song song**: vế phải được tính xong hết rồi mới gán, nên an toàn.

## 8. Cách 3 — Stack nửa đầu

Dùng nhanh/chậm; trong lúc `slow` đi, đẩy giá trị vào stack. Khi tới giữa, pop stack ra so với nửa sau. Stack giúp "đọc ngược" nửa đầu.

In [11]:
def is_palindrome_stack(head):
    stack = []
    slow = fast = head
    while fast and fast.next:
        stack.append(slow.val)
        slow = slow.next
        fast = fast.next.next
    if fast:                 # độ dài lẻ: bỏ qua node giữa
        slow = slow.next
    while slow:
        if stack.pop() != slow.val:
            return False
        slow = slow.next
    return True

for arr in [[1, 2, 2, 1], [1, 2], [1, 2, 3, 2, 1], [7], [1, 2, 3, 1]]:
    print(arr, "->", is_palindrome_stack(build(arr)))

[1, 2, 2, 1] -> True
[1, 2] -> False
[1, 2, 3, 2, 1] -> True
[7] -> True
[1, 2, 3, 1] -> False


`if fast:` sau vòng lặp: nếu `fast` khác `None` nghĩa là độ dài lẻ (fast dừng ở node cuối), node `slow` lúc đó là node giữa, cần bỏ qua. Space `O(n/2) = O(n)`.

## 9. Cách 4 — Đệ quy với con trỏ "từ đầu"

Đệ quy đi tới cuối danh sách; khi **quay về** (unwinding), các node được thăm theo thứ tự ngược. Song song, ta giữ một con trỏ `front` tiến từ đầu. So sánh `front.val` với node hiện tại khi quay về.

In [12]:
def is_palindrome_recursive(head):
    front = head

    def check(node):
        nonlocal front           # cho phép gán lại biến front của hàm ngoài
        if node is None:
            return True
        if not check(node.next): # đi tới cuối trước
            return False
        if front.val != node.val:
            return False
        front = front.next       # tiến front khi quay về
        return True

    return check(head)

for arr in [[1, 2, 2, 1], [1, 2], [1, 2, 3, 2, 1], [7]]:
    print(arr, "->", is_palindrome_recursive(build(arr)))

[1, 2, 2, 1] -> True
[1, 2] -> False
[1, 2, 3, 2, 1] -> True
[7] -> True


- `nonlocal front`: hàm lồng `check` muốn **gán lại** biến `front` của hàm ngoài; không có `nonlocal`, Python coi `front` là biến cục bộ mới → lỗi.
- Space `O(n)` cho call stack; với `n = 10^5` sẽ vượt giới hạn đệ quy mặc định (~1000) của Python → **không nên nộp** cách này trên LeetCode.

In [13]:
import sys
print("Giới hạn đệ quy mặc định:", sys.getrecursionlimit())
try:
    is_palindrome_recursive(build([1] * 5000))
except RecursionError as e:
    print("RecursionError với 5000 node:", str(e)[:50])

Giới hạn đệ quy mặc định: 1000
RecursionError với 5000 node: maximum recursion depth exceeded


## 10. Code nộp LeetCode

Bản chính `O(1)` bộ nhớ (có khôi phục danh sách):

In [14]:
class Solution:
    def isPalindrome(self, head: ListNode | None) -> bool:
        # 1) tìm giữa
        slow = fast = head
        while fast and fast.next:
            slow = slow.next
            fast = fast.next.next
        # 2) đảo nửa sau
        prev, cur = None, slow
        while cur:
            cur.next, prev, cur = prev, cur, cur.next
        # 3) so sánh
        left, right = head, prev
        ok = True
        while right:
            if left.val != right.val:
                ok = False
                break
            left, right = left.next, right.next
        # 4) khôi phục (không bắt buộc nhưng là thói quen tốt)
        p, q = None, prev
        while q:
            q.next, p, q = p, q, q.next
        return ok

sol = Solution()
print(sol.isPalindrome(build([1, 2, 2, 1])), sol.isPalindrome(build([1, 2])))

True False


**Cảnh báo về dòng gán gộp** `cur.next, prev, cur = prev, cur, cur.next`: vế phải tính trước `(prev, cur, cur.next)` — đúng giá trị cũ; rồi gán **từ trái qua phải**: `cur.next` (của `cur` cũ) trước, sau đó `prev`, cuối cùng `cur`. Nếu đổi thứ tự vế trái thành `cur, cur.next, prev = ...` thì `cur.next` sẽ gán vào node **mới** → hỏng. Người mới nên dùng bản 4 dòng với `nxt` cho chắc.

Bản ngắn dễ nhớ (O(n) bộ nhớ, vẫn Accepted):

```python
class Solution:
    def isPalindrome(self, head):
        vals = []
        while head:
            vals.append(head.val)
            head = head.next
        return vals == vals[::-1]
```

## 11. Vì sao đúng?

Gọi các giá trị là `v[0..n-1]`, `m = n // 2`. Vòng nhanh/chậm dừng với `slow` ở index `m`.

- Nửa sau đảo cho dãy `v[n-1], v[n-2], ..., v[m]` — dài `n - m` phần tử.
- Ta so sánh `v[k]` với `v[n-1-k]` cho `k = 0 .. n-m-1`. Đây chính là định nghĩa palindrome (mỗi cặp đối xứng), cộng thêm khi `n` lẻ thì cặp `k = m` là `v[m]` với chính nó — luôn khớp.
- `p1` không bao giờ `None` khi `p2` còn: `p1` đi tối đa `n - m` bước, mà nửa đầu (kể cả khi `slow` dùng chung) có đủ ≥ `n - m` node theo chuỗi `next` (với `n` lẻ, node cuối nửa đầu trỏ vào node giữa).

**Invariant vòng so sánh:** trước mỗi vòng, mọi cặp đã so đều bằng nhau; `p1` ở `v[k]`, `p2` ở `v[n-1-k]`.

## 12. Độ phức tạp

`n` = số node.

| Cách | Time | Space phụ | Ghi chú |
|---|---|---|---|
| Chép ra list | `O(n)` | `O(n)` | dễ nhất |
| Stack nửa đầu | `O(n)` | `O(n)` | học nhanh/chậm |
| Đệ quy | `O(n)` | `O(n)` call stack | dễ RecursionError |
| Đảo nửa sau | `O(n)` | **`O(1)`** | đáp án Follow-up |

Cách đảo nửa sau duyệt: `n/2` (tìm giữa) + `n/2` (đảo) + `n/2` (so) + `n/2` (khôi phục) ≈ `2n` bước → `O(n)`.

## 13. Bộ test

In [15]:
def check(name, actual, expected):
    assert actual == expected, (name, actual, expected)
    print(f"PASS  {name}")

sol = Solution()
check("ví dụ 1 [1,2,2,1]", sol.isPalindrome(build([1, 2, 2, 1])), True)
check("ví dụ 2 [1,2]", sol.isPalindrome(build([1, 2])), False)
check("một node", sol.isPalindrome(build([5])), True)
check("hai node bằng nhau", sol.isPalindrome(build([3, 3])), True)
check("lẻ palindrome", sol.isPalindrome(build([1, 2, 3, 2, 1])), True)
check("lẻ không palindrome", sol.isPalindrome(build([1, 2, 3, 1, 1])), False)
check("khác ở giữa (chẵn)", sol.isPalindrome(build([1, 2, 3, 1])), False)
check("toàn 0", sol.isPalindrome(build([0] * 10)), True)
h = build([1, 0, 0, 1]); sol.isPalindrome(h)
check("danh sách được khôi phục", to_list(h), [1, 0, 0, 1])
big = build([i % 10 for i in range(50000)] + [i % 10 for i in reversed(range(50000))])
check("10^5 node", sol.isPalindrome(big), True)

PASS  ví dụ 1 [1,2,2,1]
PASS  ví dụ 2 [1,2]
PASS  một node
PASS  hai node bằng nhau
PASS  lẻ palindrome
PASS  lẻ không palindrome
PASS  khác ở giữa (chẵn)
PASS  toàn 0
PASS  danh sách được khôi phục
PASS  10^5 node


In [16]:
import random
rng = random.Random(234)
solvers = [is_palindrome_copy, is_palindrome_copy_two_pointers, is_palindrome_reverse_half,
           is_palindrome_stack, is_palindrome_recursive, Solution().isPalindrome]
for _ in range(1000):
    n = rng.randint(1, 12)
    half = [rng.randint(0, 2) for _ in range(n // 2)]
    if rng.random() < 0.5:     # tạo palindrome chắc chắn
        arr = half + ([rng.randint(0, 2)] if n % 2 else []) + half[::-1]
    else:
        arr = [rng.randint(0, 2) for _ in range(n)]
    expected = arr == arr[::-1]
    for f in solvers:
        h = build(arr)
        assert f(h) == expected, (f.__name__, arr)
print("1000 danh sách ngẫu nhiên: 6 cài đặt đều khớp — PASS")

1000 danh sách ngẫu nhiên: 6 cài đặt đều khớp — PASS


## 14. Lỗi người mới hay gặp

### Lỗi 1: Điều kiện vòng nhanh/chậm thiếu `fast.next`

In [17]:
def wrong_middle(head):
    slow = fast = head
    while fast:                  # thiếu fast.next
        slow = slow.next
        fast = fast.next.next
    return slow

try:
    wrong_middle(build([1, 2, 3]))
except AttributeError as e:
    print("AttributeError:", e)

AttributeError: 'NoneType' object has no attribute 'next'


**Vì sao sai:** với độ dài lẻ, `fast` dừng ở node cuối, `fast.next` là `None`, `None.next` gây lỗi. **Sửa:** `while fast and fast.next:`.

### Lỗi 2: Quên lưu `next` khi đảo

In [18]:
def wrong_reverse(head):
    prev, cur = None, head
    while cur:
        cur.next = prev          # mất đường tới phần còn lại!
        prev = cur
        cur = cur.next           # cur.next giờ là prev -> lùi lại
    return prev

print("Đảo sai [1,2,3] ->", show(wrong_reverse(build([1, 2, 3]))))

Đảo sai [1,2,3] -> 1


**Vì sao sai:** sau `cur.next = prev`, `cur.next` đã bị ghi đè nên `cur = cur.next` quay lại `prev` (=`None` ở vòng đầu) → vòng lặp dừng sớm, chỉ còn 1 node. **Sửa:** lưu `nxt = cur.next` trước.

### Lỗi 3: So sánh node thay vì giá trị

`if p1 != p2` so sánh **hai đối tượng node** (địa chỉ), luôn khác nhau dù cùng `val`. Phải so `p1.val != p2.val`.

In [19]:
x, y = ListNode(4), ListNode(4)
print("x == y ?", x == y, "| x.val == y.val ?", x.val == y.val)

x == y ? False | x.val == y.val ? True


### Lỗi 4: Dừng vòng so sánh theo `p1`

Nếu viết `while p1:` thì với cách đảo nửa sau, `p1` đi qua node giữa rồi tới nửa sau (đã đảo, `slow.next = None`)... dễ so sai hoặc `AttributeError` khi `p2` hết trước. Luôn dừng theo **nửa ngắn hơn** — ở đây là `p2`.

### Lỗi 5: Không khôi phục danh sách
LeetCode vẫn Accepted, nhưng trong phỏng vấn/thực tế, hàm "kiểm tra" mà làm hỏng dữ liệu đầu vào là **side effect** xấu. Nên nhắc tới và khôi phục.

## 15. Mẹo nhận dạng pattern

- Linked list + "giữa", "nửa", "chu trình" → **nhanh/chậm (fast & slow pointers)**.
- Linked list + "đọc ngược", "so từ hai đầu" → **đảo một phần tại chỗ** hoặc stack.
- Follow-up `O(1)` bộ nhớ trên linked list → gần như luôn là "đảo tại chỗ" + "nhanh/chậm".
- Ba kỹ thuật nền cần thuộc lòng: duyệt, tìm giữa, đảo ngược.

## 16. Biến thể

- **Reverse Linked List (206)**: chính hàm `reverse_list`.
- **Middle of the Linked List (876)**: chính `find_middle`.
- **Reorder List (143)**: tìm giữa + đảo nửa sau + **trộn xen kẽ**.

In [20]:
def reorder_list(head):
    """L0->Ln->L1->Ln-1->... (LeetCode 143)."""
    if not head or not head.next:
        return head
    slow = fast = head
    while fast.next and fast.next.next:      # slow dừng ở cuối nửa đầu
        slow = slow.next
        fast = fast.next.next
    second = reverse_list(slow.next)
    slow.next = None
    first = head
    while second:
        n1, n2 = first.next, second.next
        first.next = second
        second.next = n1
        first, second = n1, n2
    return head

print(show(reorder_list(build([1, 2, 3, 4, 5]))))
print(show(reorder_list(build([1, 2, 3, 4]))))

1 -> 5 -> 2 -> 4 -> 3
1 -> 4 -> 2 -> 3


## 17. Bài tập tự luyện (TODO)

### Bài 1 — Trace tay
Với `1 -> 3 -> 3 -> 2`: `slow` dừng ở đâu? Nửa sau đảo là gì? Cặp nào khác nhau đầu tiên?

### Bài 2 — Đếm độ dài linked list

In [21]:
def exercise_length(head) -> int:
    # TODO: duyệt while node, đếm
    pass

# assert exercise_length(build([1, 2, 3])) == 3

### Bài 3 — Tự viết `reverse_list`

In [22]:
def exercise_reverse(head):
    # TODO: prev, cur, nxt — 4 dòng trong vòng lặp
    pass

# assert to_list(exercise_reverse(build([1, 2, 3]))) == [3, 2, 1]

### Bài 4 — Tự viết `isPalindrome` O(1) bộ nhớ

In [23]:
def exercise_is_palindrome(head) -> bool:
    # TODO 1: nhanh/chậm tìm giữa
    # TODO 2: đảo nửa sau
    # TODO 3: so sánh theo p2
    pass

# assert exercise_is_palindrome(build([1, 2, 2, 1])) == True
# assert exercise_is_palindrome(build([1, 2])) == False

### Bài 5 — Palindrome sau khi bỏ tối đa 1 node?
Cho danh sách, trả `True` nếu **xóa tối đa một node** có thể làm nó thành palindrome. (Được phép chép ra list.) Ví dụ `[1,2,3,1]` → `True` (bỏ 2 hoặc 3), `[1,2,3,4]` → `False`.

In [24]:
def exercise_almost_palindrome(head) -> bool:
    # TODO: chép ra list; hai con trỏ; khi lệch thử bỏ trái hoặc bỏ phải
    pass

# assert exercise_almost_palindrome(build([1, 2, 3, 1])) == True
# assert exercise_almost_palindrome(build([1, 2, 3, 4])) == False

### Bài 6 — Mini challenge: Reorder List tự viết
Viết lại `reorder_list` mà không xem mục 16, kiểm tra với `[1,2,3,4,5]` → `[1,5,2,4,3]`.

In [25]:
def exercise_reorder(head):
    # TODO: tìm giữa, cắt đôi, đảo nửa sau, trộn xen kẽ
    pass

# assert to_list(exercise_reorder(build([1, 2, 3, 4, 5]))) == [1, 5, 2, 4, 3]

## 18. Gợi ý 3 mức, đáp án, template và checklist

| Bài | Mức 1 | Mức 2 | Mức 3 |
|---|---|---|---|
| 1 | Chẵn → slow ở đầu nửa sau | slow ở node 3 thứ hai | nửa sau đảo `2 -> 3`; cặp (1,2) khác |
| 2 | Mẫu duyệt chuẩn | `count += 1` | `while node: count += 1; node = node.next` |
| 3 | Lưu `nxt` trước | 4 phép gán | `return prev` |
| 4 | Ghép mục 4 + 5 + 6 | `while p2:` | so `.val` |
| 5 | Two pointers trên list | hàm phụ `is_pal(i, j)` | `is_pal(i+1, j) or is_pal(i, j-1)` |
| 6 | Vòng `fast.next and fast.next.next` | `slow.next = None` cắt đôi | lưu `n1, n2` trước khi nối |

### Đáp án tham khảo

Bài 1: `slow` ở `3` thứ hai (index 2); nửa sau `3 -> 2` đảo thành `2 -> 3`; so `1` với `2` → khác → `False`.

In [26]:
def answer_length(head):
    count, node = 0, head
    while node:
        count += 1
        node = node.next
    return count

def answer_reverse(head):
    prev, cur = None, head
    while cur:
        nxt = cur.next
        cur.next = prev
        prev, cur = cur, nxt
    return prev

def answer_is_palindrome(head):
    slow = fast = head
    while fast and fast.next:
        slow, fast = slow.next, fast.next.next
    p1, p2 = head, answer_reverse(slow)
    while p2:
        if p1.val != p2.val:
            return False
        p1, p2 = p1.next, p2.next
    return True

def answer_almost_palindrome(head):
    vals = to_list(head)
    def is_pal(i, j):
        while i < j:
            if vals[i] != vals[j]:
                return False
            i, j = i + 1, j - 1
        return True
    i, j = 0, len(vals) - 1
    while i < j:
        if vals[i] != vals[j]:
            return is_pal(i + 1, j) or is_pal(i, j - 1)
        i, j = i + 1, j - 1
    return True

def answer_reorder(head):
    return reorder_list(head)

assert answer_length(build([1, 2, 3])) == 3
assert to_list(answer_reverse(build([1, 2, 3]))) == [3, 2, 1]
assert answer_reverse(None) is None
assert answer_is_palindrome(build([1, 2, 2, 1])) and not answer_is_palindrome(build([1, 2]))
assert not answer_is_palindrome(build([1, 3, 3, 2]))
assert answer_almost_palindrome(build([1, 2, 3, 1])) and not answer_almost_palindrome(build([1, 2, 3, 4]))
assert to_list(answer_reorder(build([1, 2, 3, 4, 5]))) == [1, 5, 2, 4, 3]
for _ in range(300):
    arr = [rng.randint(0, 1) for _ in range(rng.randint(1, 9))]
    assert answer_is_palindrome(build(arr)) == (arr == arr[::-1])
    brute = any((arr[:k] + arr[k+1:]) == (arr[:k] + arr[k+1:])[::-1] for k in range(len(arr))) or arr == arr[::-1]
    assert answer_almost_palindrome(build(arr)) == brute
print("Đáp án tham khảo: tất cả test PASS")

Đáp án tham khảo: tất cả test PASS


### Template nhớ nhanh

```python
# tìm giữa
slow = fast = head
while fast and fast.next:
    slow, fast = slow.next, fast.next.next
# đảo từ slow
prev, cur = None, slow
while cur:
    nxt = cur.next; cur.next = prev; prev = cur; cur = nxt
# so sánh
p1, p2 = head, prev
while p2:
    if p1.val != p2.val: return False
    p1, p2 = p1.next, p2.next
return True
```

### Checklist submit

- [ ] `while fast and fast.next` (không thiếu `fast.next`).
- [ ] Khi đảo, lưu `nxt` trước khi đổi `cur.next`.
- [ ] Vòng so sánh dừng theo nửa sau (`p2`).
- [ ] So sánh `.val`, không so node.
- [ ] Danh sách 1 node trả `True`.
- [ ] (Nên) khôi phục danh sách sau khi kiểm tra.
- [ ] Không dùng đệ quy sâu với `n = 10^5`.